### 4.4 Definição do Hardware

In [ ]:
# Verificar o status da GPU
!nvidia-smi

### 4.5 Configuração do Ambiente (YOLOv5 e Dependências)

In [ ]:
# Garantir que estamos no diretório base e que não existe um yolov5 anterior
%cd /content
!rm -rf yolov5  # Remove qualquer clone anterior

# Clonar o repositório YOLOv5
!git clone https://github.com/ultralytics/yolov5
%cd yolov5

# Instalar dependências do YOLOv5
%pip install -qr requirements.txt
%pip install -q roboflow

### Conexão com a Nuvem (Google Drive)

In [ ]:
from google.colab import drive
import os

# Montar o Google Drive
drive.mount('/content/drive')

# Desmontar o Google Drive
# drive.flush_and_unmount()

### 4.3 Pré-processamento dos Datasets (Pipeline MLOps Automático)

In [ ]:
import shutil
import yaml
import os

# 1. Caminhos
caminho_zip_drive = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Yolo_dataset.zip"
pasta_local_vm = "/content/Yolo_dataset"
yaml_path = f"{pasta_local_vm}/data.yaml"

# 2. Criar a pasta para não espalhar os arquivos
os.makedirs(pasta_local_vm, exist_ok=True)

# 3. Automação MLOps (Verifica se a pasta de imagens existe)
if not os.path.exists(f"{pasta_local_vm}/images"):
    print("🚚 1. Copiando o ZIP do Drive para o SSD da VM...")
    shutil.copy2(caminho_zip_drive, "/content/Yolo_dataset.zip")
    
    print("📦 2. Descompactando na velocidade da luz...")
    shutil.unpack_archive("/content/Yolo_dataset.zip", pasta_local_vm)
    print(" ✅ Descompactação concluída.")
else:
    print("✅ O dataset já está descompactado no SSD desta sessão!")

# 4. A Cirurgia do YAML (Só atua se o caminho estiver errado!)
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    # Ele confere: O caminho atual já é o da VM local?
    if data.get('path') != pasta_local_vm:
        print("🔧 3. Ajustando o caminho no data.yaml existente...")
        data['path'] = pasta_local_vm
        with open(yaml_path, 'w') as f:
            yaml.dump(data, f, sort_keys=False)
        print(" 🚀 data.yaml atualizado com sucesso!")
else:
    print(" ⚠️ data.yaml não encontrado! O YOLO vai precisar dele para rodar.")

# 5. Aponta a variável global do YOLO para o SSD local
os.environ["DATASET_DIRECTORY"] = pasta_local_vm

### 4.6 Definição de Hiperparâmetros e 4.7 Treinamento do Modelo

In [ ]:
import os
import torch
from datetime import datetime

# 1. Verificar se o PyTorch reconhece a GPU (Vital para garantir que o Colab te deu a placa de vídeo de volta)
print(f"Setup completo. Usando torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'ATENÇÃO: MODO CPU!'})")

# 2. Destrói arquivos de cache corrompidos (Garante que o YOLO leia o dataset do zero ao retomar)
!rm -f "$DATASET_DIRECTORY/labels/train/*.cache"
!rm -f "$DATASET_DIRECTORY/labels/val/*.cache"
!rm -f "$DATASET_DIRECTORY/images/train/*.cache"
!rm -f "$DATASET_DIRECTORY/images/val/*.cache"

# 3. Desativa o Weights & Biases
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
print("W&B desativado para este notebook.")

# 4. Gera uma string com a data e hora exata do início do treino (Model Registry)
agora = datetime.now()
nome_treino = agora.strftime("Treino_%Y-%m-%d_%Hh%M")
os.environ["NOME_TREINO"] = nome_treino
print(f"🚀 Os resultados deste treino serão salvos em: runs/train/{nome_treino}")

# 5. Executa o treinamento com a variável de ambiente customizada
!python train.py --batch 64 --epochs 25 --data "$DATASET_DIRECTORY/data.yaml" --weights yolov5s.pt --workers 4 --cache disk --name "$NOME_TREINO"

### Backup de Treino (Salvamento dos Pesos)

In [ ]:
import shutil
import os

# 1. Recupera o nome exato que foi usado no treinamento
nome_treino = os.environ.get("NOME_TREINO", "exp")

# 2. Caminhos dinâmicos (Origem e Destino)
origem = f"/content/yolov5/runs/train/{nome_treino}"
destino = f"/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_YOLOv5/{nome_treino}"

# 3. Copia a pasta inteira, sobrescrevendo arquivos velhos se necessário
shutil.copytree(origem, destino, dirs_exist_ok=True)

print(f"✅ Todos os gráficos e o modelo foram salvos em: {destino}")

### 4.7 Inferência com o Conjunto de Testes

In [ ]:
import os

# 1. Recupera o nome exato da pasta do modelo treinado
nome_treino = os.environ.get("NOME_TREINO", "exp")

# 2. Constrói os caminhos no Python
caminho_modelo_drive = f"/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_YOLOv5/{nome_treino}/weights/best.pt"
nome_teste = nome_treino.replace("Treino", "Teste")

print("🔍 Testando o modelo contra dados nunca vistos (Conjunto de Teste)...")
print(f"Caminho do Modelo: {caminho_modelo_drive}")

# 3. Transforma as variáveis do Python em Variáveis de Ambiente do Linux
os.environ["CAMINHO_MODELO"] = caminho_modelo_drive
os.environ["NOME_TESTE"] = nome_teste

# 4. O Linux agora reconhece o cifrão perfeitamente e faz o teste!
!python val.py --weights "$CAMINHO_MODELO" --data "$DATASET_DIRECTORY/data.yaml" --task test --name "$NOME_TESTE"

### Backup do teste


In [ ]:
import shutil
import os

# 1. Recupera as variáveis de nome
nome_treino = os.environ.get("NOME_TREINO", "exp")
nome_teste = nome_treino.replace("Treino", "Teste")

# 2. Define a origem (VM) e o destino (Drive)
pasta_resultados_teste_vm = f"/content/yolov5/runs/val/{nome_teste}"
pasta_teste_drive = f"/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_YOLOv5/{nome_teste}"

print(f"💾 Iniciando backup da pasta de teste para o Google Drive...")

# 3. Executa a cópia de segurança
shutil.copytree(pasta_resultados_teste_vm, pasta_teste_drive, dirs_exist_ok=True)

print(f"✅ Backup concluído com sucesso!")
print(f"👉 Salvo em: {pasta_teste_drive}")

### 4.8 Coleta De metricas visuais

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

def mostrar_metrica(caminho_imagem, titulo):
    """Lê uma imagem do disco e a exibe no Colab."""
    img = cv2.imread(caminho_imagem)
    if img is not None:
        plt.figure(figsize=(15, 10))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(titulo, fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.show()
    else:
        print(f"⚠️ Métrica não encontrada no Drive: {caminho_imagem}")

# 1. Recupera os nomes
nome_treino = os.environ.get("NOME_TREINO", "exp")
nome_teste = nome_treino.replace("Treino", "Teste")

# 2. AGORA TUDO VEM DO SEU DRIVE! (Segurança Máxima)
pasta_treino_drive = f"/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_YOLOv5/{nome_treino}"
pasta_teste_drive = f"/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_YOLOv5/{nome_teste}"

print(f"📊 Lendo Métricas Oficiais direto do Google Drive...")
print("-" * 60)

# 3. Gráficos do Treinamento
print(">>> MÉTRICAS DE TREINAMENTO (Acurácia e Loss)")
mostrar_metrica(f"{pasta_treino_drive}/results.png", "Acompanhamento do Treinamento")

# 4. Gráficos do Teste
print("\n>>> MÉTRICAS DE TESTE (Avaliação Final)")
mostrar_metrica(f"{pasta_teste_drive}/confusion_matrix.png", "Matriz de Confusão")
mostrar_metrica(f"{pasta_teste_drive}/PR_curve.png", "Curva Precision-Recall")

# 5. Visualização Prática
print("\n>>> INFERÊNCIA PRÁTICA (Ground Truth vs Predições)")
mostrar_metrica(f"{pasta_teste_drive}/val_batch0_labels.jpg", "Anotações Reais (Ground Truth)")
mostrar_metrica(f"{pasta_teste_drive}/val_batch0_pred.jpg", "Predições da IA (O que ela adivinhou)")

print("\n✅ Fluxo finalizado! O seu modelo está treinado, testado e com backup garantido.")